In [ ]:
import sys
import os
sys.path.insert(0, os.path.join(os.getcwd(), 'src'))

import pandas as pd
import ee
from src.auth import AuthenticationService
from src.variables.ndvi import extract_ndvi
from src.variables.canopy_height import extract_canopy_height
from src.variables.elevation import extract_elevation
from src.variables.landcover import extract_landcover
from src.variables.worldclim import extract_worldclim
from src.variables.biomass import extract_biomass
from src.variables.waterdist import extract_distance_to_water
from src.variables.nighttime_lights import extract_nighttime_lights
from src.variables.bii import extract_bii
from src.variables.satellite_embedding import extract_satellite_embedding
from src.aoi import create_aoi_from_coordinates
from src.utils import plot_map, plot_images
from src.sampling import clean_coordinates_dataframe, convert_points_to_buffered_features
from src.export import export_rasters_to_gdrive, export_csv

from src.hex_grid import generate_h3_hexagons, extract_h3_values

In [ ]:
GEE_PROJECT_ID = "wide-office-411000"
LOCATIONS_CSV_PATH = "./input/configuration.csv"
OUTPUT_CSV_PATH = "output/env_vars.csv"
OUTPUT_H3_CSV_PATH = "output/hexagons_with_data.csv"
OUTPUT_H3_GPKG_PATH = "output/hexagons_with_data.gpkg"
OUTPUT_H3_SHP_PATH = "output/hexagons.shp"
LAT_COLUMN_NAME = "latitude"
LON_COLUMN_NAME = "longitude"
HEX_RES = 8
SAMPLING_POINT_BUFFER_METERS = 200
AOI_BUFFER_KM = 30
# SERVICE_ACCOUNT = "service-account@google-cloud-project.iam.gserviceaccount.com"
# SERVICE_ACCOUNT_KEY_FILE = "/abs/path/to/service-account-key.json"

In [ ]:
ok = AuthenticationService.authenticate(project_id=GEE_PROJECT_ID)
if not ok:
    raise SystemExit("Could not initialize EE")

In [ ]:
df = pd.read_csv(LOCATIONS_CSV_PATH)

df = clean_coordinates_dataframe(df,
                                 lat_col=LAT_COLUMN_NAME,
                                 lon_col=LON_COLUMN_NAME)

points_fc = convert_points_to_buffered_features(df,
                                                lat_col=LAT_COLUMN_NAME,
                                                lon_col=LON_COLUMN_NAME,
                                                buffer_meters=SAMPLING_POINT_BUFFER_METERS)

In [ ]:
fig, ax = plot_map(df, 
                   basemap='satellite', 
                   alpha=0.7, 
                   color="#1FDBDB", 
                   buffer_pct=0.1,
                   title="Site Locations")  

In [ ]:
aoi = create_aoi_from_coordinates(df,
                                  buffer_km=AOI_BUFFER_KM)

In [ ]:
# Process NDVI
df, image_ndvi = extract_ndvi(
    df=df,
    aoi=aoi,
    points_feature_collection=points_fc,
    start_date="2025-01-01",
    end_date="2025-12-31",
    scale=30,
    mask_water=True,
    cloud_cover_threshold=50
)
df

In [ ]:
df, image_canopy_height = extract_canopy_height(df,
                                                aoi=aoi,
                                                points_feature_collection=points_fc,
                                                scale=10)
df

In [ ]:
df, image_biomass = extract_biomass(
    df,
    aoi=aoi,
    points_feature_collection=points_fc,
    scale=100,
    year=2022
)
df

In [ ]:
df, image_landcover = extract_landcover(
    df,
    aoi=aoi,
    points_feature_collection=points_fc,
    scale=10
)
df

In [ ]:
df, image_elevation = extract_elevation(df=df,
                                        aoi=aoi,
                                        points_feature_collection=points_fc,
                                        scale=30)
df

In [ ]:
df, image_waterdist = extract_distance_to_water(
    df,
    aoi=aoi,
    points_feature_collection=points_fc,
    scale=30,
    max_search_distance=20_000,
)
df

In [ ]:
image_waterdist = image_waterdist.select('distance_to_water_m')
image_waterdist

In [ ]:
df, image_bii = extract_bii(
    df=df,
    aoi=aoi,
    points_feature_collection=points_fc,
    year=2020,
    scale=100
)

In [ ]:
df, image_nightlights = extract_nighttime_lights(
    df,
    aoi=aoi,
    points_feature_collection=points_fc,
    scale=464,
    year=2022
)
df

In [ ]:
# Do not save embeddings images for now
df, _ = extract_satellite_embedding(
    df=df,
    aoi=aoi,
    points_feature_collection=points_fc,
    year=2024,
    scale=10
)

In [ ]:
df, image_worldclim = extract_worldclim(df,
                                        aoi=aoi,
                                        points_feature_collection=points_fc,
                                        variables=['bio01', 'bio02'], # uncomment to select specific variables
                                        )
df

In [ ]:
image_stack = ee.Image.cat([
    image_ndvi,
    image_canopy_height,
    image_biomass,
    image_landcover,
    image_elevation,
    image_waterdist,
    image_bii,
    image_nightlights,
    image_worldclim,
])

In [ ]:
map = plot_images(image_stack,
                  aoi=aoi,
                  zoom=10,
                  scale=30,
                  filter_bands=['elevation'],
                  )
map

# Creating hexagon and extraction values

In [ ]:
hexagon = generate_h3_hexagons(aoi=aoi, h3_resolution=HEX_RES)
hexagon

In [ ]:
image_dict_no_categorical = {
    'ndvi': (image_ndvi, 30),
    'canopy_height': (image_canopy_height, 10),
    'biomass': (image_biomass, 100),
    'elevation': (image_elevation.select('elevation'), 30),
    'slope': (image_elevation.select('slope_percent'), 30),
    'water_dist': (image_waterdist, 30),
    'bii': (image_bii, 100),
    'nightlights': (image_nightlights, 464),
    'bio1': (image_worldclim.select('bio01'), 1000),
    'bio12': (image_worldclim.select('bio02'), 1000),
    'landcover': (image_landcover, 10, "mode")
}

In [ ]:
hexagon.explore()

In [ ]:
hexagons_with_data = extract_h3_values(
    hex_gdf=hexagon,
    image_dict=image_dict_no_categorical,
    batch_size=1000,
    tileScale=16,
    default_reducer = 'median',
)

In [ ]:
hexagons_with_data

In [ ]:
hexagons_with_data.plot(column='ndvi', legend=False, cmap='viridis', figsize=(10, 8))


In [ ]:
hexagons_with_data.to_csv(OUTPUT_H3_CSV_PATH, index=False)
hexagons_with_data.to_file(OUTPUT_H3_GPKG_PATH)
hexagons_with_data.to_file(OUTPUT_H3_SHP_PATH)
export_csv(df, OUTPUT_CSV_PATH)

In [ ]:
#export_rasters_to_gdrive(
#    image=image_stack,
#    region=aoi,
#    file_name_prefix="2025",
#    scale=200,
#    crs="EPSG:3857",
#    file_format="GeoTIFF")

___